In [13]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "Data"
RAW = ROOT / "PAMAP2_Dataset" / "Protocol"

EXPECTED = {
    "training_rows": 229015,
    "reporting_rows": 58242,
    "training_participants": [
        "subject101", "subject102", "subject105",
        "subject106", "subject107", "subject108", "subject109"
    ],
    "reporting_participants": ["subject103", "subject104"]
}

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Working directory:", ROOT)
print("Official Data/ exists:", DATA.is_dir())

Python: 3.14.4
pandas: 3.1.1
Working directory: /Users/lainegoldmacher/DS4021-Final-Project
Official Data/ exists: True


In [14]:
from itertools import combinations
from tempfile import TemporaryDirectory

if DATA.exists():
    print("Data/ already exists: SKIPPING generation to protect the reporting set.")
else:
    raw_files = sorted(RAW.glob("subject*.dat"))
    expected_raw = [f"subject{i}.dat" for i in range(101, 110)]
    if [p.name for p in raw_files] != expected_raw:
        raise FileNotFoundError(
            "Missing original PAMAP2 Protocol files. Expected subject101.dat "
            "through subject109.dat inside PAMAP2_Dataset/Protocol/."
        )

    with TemporaryDirectory(prefix="pamap2_build_", dir=ROOT) as tmp:
        stage = Path(tmp)
        all_dir = stage / "all_subjects"
        all_dir.mkdir()
        counts = {}
        for source in raw_files:
            frame = pd.read_csv(source, sep=r"\s+", header=None)
            if frame.shape[1] != 54:
                raise ValueError(f"{source.name}: expected 54 columns, got {frame.shape[1]}")
            reduced = frame.iloc[::10].copy()
            target = all_dir / (source.stem + ".csv.gz")
            reduced.to_csv(target, index=False, header=False, compression="gzip")
            counts[source.stem] = len(reduced)
            print(f"{source.stem}: {len(reduced):,} downsampled rows")
            del frame, reduced

        total = sum(counts.values())
        candidates = []
        for pair in combinations(sorted(counts), 2):
            reporting_n = sum(counts[s] for s in pair)
            candidates.append((abs(reporting_n / total - 0.2), pair, reporting_n))
        _, pair, reporting_n = min(candidates)
        reporting_ids = sorted(pair)
        training_ids = sorted(set(counts) - set(pair))
        training_n = total - reporting_n

        actual = dict(
            training_rows=training_n,
            reporting_rows=reporting_n,
            training_participants=training_ids,
            reporting_participants=reporting_ids,
        )
        if actual != EXPECTED:
            raise ValueError(
                "The newly computed split does not match the finalized project split. "
                f"Actual: {actual}. Nothing was written to Data/."
            )

        train_dir = stage / "training_set"
        report_dir = stage / "reporting_set"
        train_dir.mkdir()
        report_dir.mkdir()
        for subject in sorted(counts):
            target_dir = report_dir if subject in pair else train_dir
            (all_dir / (subject + ".csv.gz")).rename(target_dir / (subject + ".csv.gz"))
        all_dir.rmdir()

        metadata = {
            "dataset": "PAMAP2 Protocol, downsampled to 10 Hz",
            **actual,
            "training_percentage": round(training_n / total * 100, 2),
            "reporting_percentage": round(reporting_n / total * 100, 2),
        }
        (stage / "split_info.json").write_text(json.dumps(metadata, indent=2))

        if DATA.exists():
            raise FileExistsError("Data/ appeared during preparation; refusing to overwrite.")
        DATA.mkdir()
        for item in list(stage.iterdir()):
            item.rename(DATA / item.name)
        print("Created Data/ with the official split.")


Data/ already exists: SKIPPING generation to protect the reporting set.


In [15]:
if not DATA.is_dir():
    raise FileNotFoundError("Data/ not found.")

with (DATA / "split_info.json").open() as f:
    metadata = json.load(f)

for key, expected in EXPECTED.items():
    assert metadata[key] == expected, (
        f"Unexpected {key}: {metadata[key]} (expected {expected})"
    )

def names_in(folder):
    return sorted(p.name.removesuffix(".csv.gz")
                  for p in folder.glob("subject*.csv.gz"))

assert names_in(DATA / "training_set") == EXPECTED["training_participants"]
assert names_in(DATA / "reporting_set") == EXPECTED["reporting_participants"]

total = metadata["training_rows"] + metadata["reporting_rows"]
print("Dataset is correctly organized.")
print("Total:", f"{total:,}")
print("Training:", f"{metadata['training_rows']:,}", f"({metadata['training_rows']/total:.2%})")
print("Reporting:", f"{metadata['reporting_rows']:,}", f"({metadata['reporting_rows']/total:.2%})")
print("No reporting-set measurements were opened or modified.")


Dataset is correctly organized.
Total: 287,257
Training: 229,015 (79.72%)
Reporting: 58,242 (20.28%)
No reporting-set measurements were opened or modified.
